# HW 4 — Regex with AI: Generate It, Then Verify It

## Overview
This notebook uses an AI-generated regular expression to extract subsidiary names and locations from Federal Signal Corporation's FY2024 SEC Exhibit 21. The result is then verified against manually established ground truth and several edge cases.

## Part A — Generate It

**Goal:** download the SEC Exhibit 21, inspect the raw HTML, document the AI prompt/code, and run the AI-generated regex.

### A1. Student and Filing Information

In [25]:
# --- Cell 1: you and your claimed filing ---

name        = "리오디노 라이한"
student_id  = "50261893"
company     = "Federal Signal Corporation"
fiscal_year = "FY2024"
claimed_url = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

print("Name:", name)
print("Student ID:", student_id)
print("Company:", company)
print("Fiscal Year:", fiscal_year)
print("Exhibit 21 URL:", claimed_url)

Name: 리오디노 라이한
Student ID: 50261893
Company: Federal Signal Corporation
Fiscal Year: FY2024
Exhibit 21 URL: https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm


### A2. Download SEC Exhibit 21

In [26]:
# --- Cell 2: fetch the exhibit ---

import requests

URL = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

headers = {
    "User-Agent": "Riodino Raihan riodinoraihan@gmail.com"
}

response = requests.get(URL, headers=headers)

print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))

html = response.text

Status code: 200
Downloaded characters: 21884


### A3. Inspect the Downloaded HTML

The first part of the downloaded HTML is displayed below to confirm that the SEC Exhibit 21 was successfully retrieved.

In [27]:
print(html[:500])

<DOCUMENT>
<TYPE>EX-21
<SEQUENCE>3
<FILENAME>fss-20241231x10kexhx21.htm
<DESCRIPTION>SUBSIDIARIES OF THE REGISTRANT
<TEXT>
<html><head>
<!-- Document created using Wdesk -->
<!-- Copyright 2025 Workiva -->
<title>Document</title></head><body><div id="i0b29d3f9341a44a1b83bbcc9e50cc3e3_1"></div><div style="min-height:42.75pt;width:100%"><div><font><br></font></div></div><div style="text-align:right"><font style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:700;


### A4. AI Tool, Exact Prompt, and Exact AI-Generated Code

**AI Tool / Model:** Gemini 3.1 Pro

**Exact Prompt:**

The following is the exact prompt used to generate the regular expression code:

> I am doing an assignment that requires me to use regular expressions to extract every (Subsidiary Name, Location) pair from a specific SEC Exhibit 21 HTML document.
>
> The document is Federal Signal Corporation, FY2024:
>
> https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm
>
> In my Python/Colab code, the downloaded raw HTML is stored in a variable named `html`.
>
> Please write Python code using regular expressions to extract every subsidiary name and its jurisdiction/location from the table. The result should be a list of tuples in the form `(Subsidiary Name, Location)`.
>
> The HTML may contain multiple `<td>` elements in each table row, so identify the first cell as the subsidiary name and the last relevant cell as the jurisdiction/location. Clean HTML tags, HTML entities such as `&amp;`, and surrounding or repeated whitespace.
>
> Please return the complete Python code that I can run directly on the `html` variable. Also print the total number of extracted pairs and the first 10 pairs.
>
> Do not use BeautifulSoup or pandas; this part must use regular expressions.

**Exact AI-Generated Code:**

```python
import re
import html as html_lib

# (Assuming the `html` variable is already defined and contains the raw HTML string)
# html = "..."

def extract_subsidiaries(html_content):
    # Regex patterns for finding rows, cells, and removing HTML tags
    # We account for possible whitespace in closing tags (e.g., </tr >)
    row_regex = re.compile(r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>', re.IGNORECASE | re.DOTALL)
    cell_regex = re.compile(r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>', re.IGNORECASE | re.DOTALL)
    tag_regex = re.compile(r'<[^>]+>')
    space_regex = re.compile(r'\s+')
    
    extracted_pairs = []
    
    # 1. Extract all <tr> blocks
    rows = row_regex.findall(html_content)
    
    for row in rows:
        # 2. Extract all <td> blocks within the current row
        cells = cell_regex.findall(row)
        
        cleaned_cells = []
        for cell in cells:
            # Replace HTML tags with a space to prevent words from fusing together
            text = tag_regex.sub(' ', cell)
            
            # Unescape HTML entities (converts &amp; to &, &#160; to space, etc.)
            text = html_lib.unescape(text)
            
            # Replace multiple whitespace characters (newlines, tabs, etc.) with a single space
            text = space_regex.sub(' ', text).strip()
            
            # If the cell contains text after cleaning, add it to our valid cells list
            if text:
                cleaned_cells.append(text)
        
        # 3. Process rows with at least 2 non-empty cells
        if len(cleaned_cells) >= 2:
            # First relevant cell is the Subsidiary Name, the last is the Location
            subsidiary_name = cleaned_cells[0]
            location = cleaned_cells[-1]
            
            # Ignore table header rows that usually contain words like "Subsidiary", "Name", "Jurisdiction"
            name_lower = subsidiary_name.lower()
            loc_lower = location.lower()
            if ("subsidiary" in name_lower or "name of" in name_lower or 
                "jurisdiction" in loc_lower or "state" in loc_lower):
                continue
                
            extracted_pairs.append((subsidiary_name, location))
            
    return extracted_pairs

# --- Execution ---
pairs = extract_subsidiaries(html)

# --- Output Results ---
print(f"Total extracted pairs: {len(pairs)}\n")

print("First 10 pairs:")
for i, (name, loc) in enumerate(pairs[:10], start=1):
    print(f"{i}. {name} | {loc}")
```


### A5. Run the AI-Generated Regex

In [28]:
import re
import html as html_lib

# (Assuming the `html` variable is already defined and contains the raw HTML string)
# html = "..."

def extract_subsidiaries(html_content):
    # Regex patterns for finding rows, cells, and removing HTML tags
    # We account for possible whitespace in closing tags (e.g., </tr >)
    row_regex = re.compile(r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>', re.IGNORECASE | re.DOTALL)
    cell_regex = re.compile(r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>', re.IGNORECASE | re.DOTALL)
    tag_regex = re.compile(r'<[^>]+>')
    space_regex = re.compile(r'\s+')
    
    extracted_pairs = []
    
    # 1. Extract all <tr> blocks
    rows = row_regex.findall(html_content)
    
    for row in rows:
        # 2. Extract all <td> blocks within the current row
        cells = cell_regex.findall(row)
        
        cleaned_cells = []
        for cell in cells:
            # Replace HTML tags with a space to prevent words from fusing together
            text = tag_regex.sub(' ', cell)
            
            # Unescape HTML entities (converts &amp; to &, &#160; to space, etc.)
            text = html_lib.unescape(text)
            
            # Replace multiple whitespace characters (newlines, tabs, etc.) with a single space
            text = space_regex.sub(' ', text).strip()
            
            # If the cell contains text after cleaning, add it to our valid cells list
            if text:
                cleaned_cells.append(text)
        
        # 3. Process rows with at least 2 non-empty cells
        if len(cleaned_cells) >= 2:
            # First relevant cell is the Subsidiary Name, the last is the Location
            subsidiary_name = cleaned_cells[0]
            location = cleaned_cells[-1]
            
            # Ignore table header rows that usually contain words like "Subsidiary", "Name", "Jurisdiction"
            name_lower = subsidiary_name.lower()
            loc_lower = location.lower()
            if ("subsidiary" in name_lower or "name of" in name_lower or 
                "jurisdiction" in loc_lower or "state" in loc_lower):
                continue
                
            extracted_pairs.append((subsidiary_name, location))
            
    return extracted_pairs

# --- Execution ---
pairs = extract_subsidiaries(html)

# --- Output Results ---
print(f"Total extracted pairs: {len(pairs)}\n")

print("First 10 pairs:")
for i, (name, loc) in enumerate(pairs[:10], start=1):
    print(f"{i}. {name} | {loc}")

Total extracted pairs: 33

First 10 pairs:
1. Crysteel Manufacturing, Inc. | Minnesota
2. Deist Industries, LLC | Delaware
3. Elgin Sweeper Company | Delaware
4. Federal Signal of Texas Corp. | Texas
5. Federal Signal UK Holdings Limited | United Kingdom
6. Federal Signal VAMA, S.A. | Spain
7. FS Depot, LLC | Wisconsin
8. FST Canada Inc. | Canada
9. FST of Tennessee, Inc. | Tennessee
10. GenNx/TBEI Intermediate Co. | Delaware


## Part B — Verify It

**Goal:** establish ground truth and check whether the AI-generated extraction is correct.

### B1. Ground Truth

I manually counted the named subsidiary rows in the Exhibit 21 table. The manual count is **33 subsidiaries**, excluding the table header and the footnote below the table.

**Manual count = 33**

**AI-generated regex count = 33**

Therefore, the counts match.

### B2. Count Verification

In [29]:
print("Manual count:", 33)
print("AI-generated regex count:", len(pairs))
print("Counts match:", len(pairs) == 33)

Manual count: 33
AI-generated regex count: 33
Counts match: True


### B3. Complete Extraction Result

In [30]:
print("All extracted pairs:")
for i, pair in enumerate(pairs, start=1):
    print(f"{i}. {pair[0]} | {pair[1]}")

All extracted pairs:
1. Crysteel Manufacturing, Inc. | Minnesota
2. Deist Industries, LLC | Delaware
3. Elgin Sweeper Company | Delaware
4. Federal Signal of Texas Corp. | Texas
5. Federal Signal UK Holdings Limited | United Kingdom
6. Federal Signal VAMA, S.A. | Spain
7. FS Depot, LLC | Wisconsin
8. FST Canada Inc. | Canada
9. FST of Tennessee, Inc. | Tennessee
10. GenNx/TBEI Intermediate Co. | Delaware
11. Ground Force Manufacturing LLC | Delaware
12. Guzzler Manufacturing, Inc. | Alabama
13. HighMark Traffic Services, Inc. | Montana
14. Jetstream of Houston, Inc. | Delaware
15. Jetstream of Houston LLP | Texas
16. Joe Johnson Equipment LLC | Delaware
17. Mark Rite Lines Equipment Company, Inc. | Delaware
18. Northend Truck Equipment, LLC | Washington
19. OSW Equipment & Repair, LLC | Washington
20. Ox Bodies, Inc. | Alabama
21. Rugby Manufacturing Company | Oregon
22. Tishomingo Acquisition, LLC | Delaware
23. Travis Acquisition LLC | Delaware
24. Travis Body and Trailer, Inc. | Tex

### B4. First and Last Row Verification

In [31]:
print("First extracted row:")
print(pairs[0])

print("\nLast extracted row:")
print(pairs[-1])

First extracted row:
('Crysteel Manufacturing, Inc.', 'Minnesota')

Last extracted row:
('Work Equipment Ltd.', 'Canada')


### B5. Awkward Row Tests

These tests check subsidiary names containing characters that can sometimes cause extraction problems, including `&` and parentheses.

**Test 1 — Ampersand (`&`)**

In [32]:
for pair in pairs:
    if "OSW Equipment" in pair[0]:
        print(pair)

('OSW Equipment & Repair, LLC', 'Washington')


**Test 2 — Ampersand (`&`) in a longer name**

In [33]:
for pair in pairs:
    if "Truck Bodies & Equipment" in pair[0]:
        print(pair)

('Truck Bodies & Equipment International, Inc.', 'Delaware')


**Test 3 — Parentheses (`(...)`)**

In [34]:
for pair in pairs:
    if "Victor Industrial Equipment" in pair[0]:
        print(pair)

('Victor Industrial Equipment (PTY) Limited', 'South Africa')


**Test 4 — HTML entity and whitespace handling**

In [35]:
# Check the raw HTML for HTML entities.
import re

entity_matches = re.findall(r'&amp;|&nbsp;|&#160;', html, flags=re.IGNORECASE)

print("HTML entity occurrences found in raw HTML:", len(entity_matches))
print("Sample entities:", entity_matches[:10])


entity_test_html = """
<table>
<tr>
    <td>Test&nbsp;Entity</td>
    <td>Delaware</td>
</tr>
</table>
"""

entity_test_pairs = extract_subsidiaries(entity_test_html)

print("\nEntity/whitespace test:")
print(entity_test_pairs)

HTML entity occurrences found in raw HTML: 2
Sample entities: ['&#160;', '&#160;']

Entity/whitespace test:
[('Test Entity', 'Delaware')]


### B6. Missing-Location Edge Case


The original Federal Signal Corporation FY2024 Exhibit 21 does not contain a named subsidiary row with a missing jurisdiction/location. Therefore, I did not invent a filing failure. Instead, I used a small synthetic HTML example with an empty location cell to test the behavior of the extraction function. In this case, the function returned an empty list rather than producing an incomplete `(Subsidiary Name, Location)` pair. This confirms that the current implementation does not create a malformed pair when the location cell is empty, although it also means that such a row would not be extracted.


In [36]:
test_html = """
<table>
<tr>
<td>Test Subsidiary</td>
<td></td>
</tr>
</table>
"""

test_pairs = extract_subsidiaries(test_html)

print("Missing-location test:")
print(test_pairs)

Missing-location test:
[]


### B7. Path 2 — Proving It

I established the ground truth manually by counting the subsidiary rows in Federal Signal Corporation's FY2024 Exhibit 21. The filing contains 33 named subsidiary rows, and the AI-generated regex also extracted 33 pairs, so the counts match exactly.

The first extracted pair was `Crysteel Manufacturing, Inc. — Minnesota`, which matches the first subsidiary row in the filing. The last extracted pair was `Work Equipment Ltd. — Canada`, which matches the last subsidiary row.

I deliberately checked awkward entries rather than only the easiest rows. `OSW Equipment & Repair, LLC — Washington` and `Truck Bodies & Equipment International, Inc. — Delaware` verified that the extraction preserved ampersands in subsidiary names. `Victor Industrial Equipment (PTY) Limited — South Africa` verified that parentheses were preserved correctly. I also inspected the raw HTML for HTML entities and whitespace handling.

The original filing does not contain a named subsidiary row with a missing location, so I did not invent a filing failure. Instead, I used a separate artificial HTML test with an empty location cell to check how the extraction function behaves in that situation. The function returned an empty list, so it did not create an incomplete subsidiary-location pair.

### B8. Explicit Comparison with Official Exhibit 21

Compare the AI-extracted subsidiaries directly against the official Exhibit 21 provided in the assignment.

In [46]:
# ===== EXPLICIT COMPARISON WITH OFFICIAL EXHIBIT 21 =====

# Official subsidiaries from Exhibit 21 that is given from task instructions
official_subsidiaries = [
    ("Crysteel Manufacturing, Inc.", "Minnesota"),
    ("Deist Industries, LLC", "Delaware"),
    ("Elgin Sweeper Company", "Delaware"),
    ("Federal Signal of Texas Corp.", "Texas"),
    ("Federal Signal UK Holdings Limited", "United Kingdom"),
    ("Federal Signal VAMA, S.A.", "Spain"),
    ("FS Depot, LLC", "Wisconsin"),
    ("FST Canada Inc.", "Canada"),
    ("FST of Tennessee, Inc.", "Tennessee"),
    ("GenNx/TBEI Intermediate Co.", "Delaware"),
    ("Ground Force Manufacturing LLC", "Delaware"),
    ("Guzzler Manufacturing, Inc.", "Alabama"),
    ("HighMark Traffic Services, Inc.", "Montana"),
    ("Jetstream of Houston, Inc.", "Delaware"),
    ("Jetstream of Houston LLP", "Texas"),
    ("Joe Johnson Equipment LLC", "Delaware"),
    ("Mark Rite Lines Equipment Company, Inc.", "Delaware"),
    ("Northend Truck Equipment, LLC", "Washington"),
    ("OSW Equipment & Repair, LLC", "Washington"),
    ("Ox Bodies, Inc.", "Alabama"),
    ("Rugby Manufacturing Company", "Oregon"),
    ("Tishomingo Acquisition, LLC", "Delaware"),
    ("Travis Acquisition LLC", "Delaware"),
    ("Travis Body and Trailer, Inc.", "Texas"),
    ("Travis Leasing, LLC", "Delaware"),
    ("Truck Bodies & Equipment International, Inc.", "Delaware"),
    ("Vactor Manufacturing, LLC", "Illinois"),
    ("Victor Industrial Equipment (PTY) Limited", "South Africa"),
    ("Victor Products Holdings Ltd.", "United Kingdom"),
    ("Victor Products Ltd.", "United Kingdom"),
    ("Victor Products USA, Incorporated", "Delaware"),
    ("Western Truck Body Mfg. ULC", "Canada"),
    ("Work Equipment Ltd.", "Canada"),
]

print("=== COMPARISON: AI-Extracted vs Official Exhibit 21 ===\n")
print(f"Official count: {len(official_subsidiaries)}")
print(f"AI-extracted count: {len(pairs)}\n")

# Check exact match
all_match = True
mismatches = []

for i, (official, extracted) in enumerate(zip(official_subsidiaries, pairs), start=1):
    if official != extracted:
        all_match = False
        mismatches.append({
            'row': i,
            'official': official,
            'extracted': extracted
        })

if all_match:
    print("✅ PERFECT MATCH: All 33 subsidiaries extracted correctly!")
    print("\nDetailed verification:")
    for i, (official, extracted) in enumerate(zip(official_subsidiaries, pairs), start=1):
        name_match = "✓" if official[0] == extracted[0] else "✗"
        loc_match = "✓" if official[1] == extracted[1] else "✗"
        print(f"{i:2d}. {name_match} {extracted[0]:<45} {loc_match} {extracted[1]}")
else:
    print("❌ MISMATCHES FOUND:\n")
    for mismatch in mismatches:
        print(f"Row {mismatch['row']}:")
        print(f"  Official:  {mismatch['official']}")
        print(f"  Extracted: {mismatch['extracted']}\n")

print("\n=== CONCLUSION ===")
print(f"Accuracy: {(len(pairs) - len(mismatches)) / len(official_subsidiaries) * 100:.1f}%")

=== COMPARISON: AI-Extracted vs Official Exhibit 21 ===

Official count: 33
AI-extracted count: 33

✅ PERFECT MATCH: All 33 subsidiaries extracted correctly!

Detailed verification:
 1. ✓ Crysteel Manufacturing, Inc.                  ✓ Minnesota
 2. ✓ Deist Industries, LLC                         ✓ Delaware
 3. ✓ Elgin Sweeper Company                         ✓ Delaware
 4. ✓ Federal Signal of Texas Corp.                 ✓ Texas
 5. ✓ Federal Signal UK Holdings Limited            ✓ United Kingdom
 6. ✓ Federal Signal VAMA, S.A.                     ✓ Spain
 7. ✓ FS Depot, LLC                                 ✓ Wisconsin
 8. ✓ FST Canada Inc.                               ✓ Canada
 9. ✓ FST of Tennessee, Inc.                        ✓ Tennessee
10. ✓ GenNx/TBEI Intermediate Co.                   ✓ Delaware
11. ✓ Ground Force Manufacturing LLC                ✓ Delaware
12. ✓ Guzzler Manufacturing, Inc.                   ✓ Alabama
13. ✓ HighMark Traffic Services, Inc.               ✓ Montana


## Part C — Fixing and Explaining It

**Goal:** determine whether a fix is necessary and explain how the regular expressions work.

### C1. Fix

No fix was needed. The original AI-generated regex successfully extracted all 33 subsidiary-location pairs from the Federal Signal Corporation FY2024 Exhibit 21. The first and last rows were correctly extracted, and the tested awkward cases were also handled correctly.

Because the verification results matched the manually established ground truth, I did not modify the AI-generated regex.

### C2. Regex Explanation

#### C2.1 Row Pattern

The row pattern is used to find each HTML table row (`<tr>...</tr>`).

In [37]:
row_regex = re.compile(
    r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>',
    re.IGNORECASE | re.DOTALL
)

**Explanation:**
- `<\s*tr` identifies an opening `<tr>` tag while allowing optional whitespace.
- `[^>]*` allows attributes inside the opening tag.
- `(.*?)` captures the contents of the row without taking more text than necessary.
- `<\s*/\s*tr\s*>` matches the closing `</tr>` tag.
- `.*?` uses a lazy match. The `?` makes `*` non-greedy, so the pattern stops at the nearest matching closing `</tr>` instead of consuming as much text as possible.
- `\s*` means zero or more whitespace characters. I use `*` rather than `+` because the whitespace is optional: `<tr>` is valid HTML, but the pattern should also tolerate formatting such as `< tr>` or whitespace around the tag syntax.
- `re.IGNORECASE` allows different capitalization of HTML tags.
- `re.DOTALL` allows the match to include line breaks.

#### C2.2 Cell Pattern

The cell pattern is used to find individual table cells (`<td>...</td>`) inside each row.

In [38]:
cell_regex = re.compile(
    r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>',
    re.IGNORECASE | re.DOTALL
)

**Explanation:**

- `<\s*td` identifies an opening `<td>` tag.
- `[^>]*` allows attributes inside the opening tag.
- `(.*?)` similarly captures the smallest amount of cell content needed before the closing `</td>`.
- `<\s*/\s*td\s*>` matches the closing `</td>` tag while allowing optional whitespace.
- `re.IGNORECASE` and `re.DOTALL` provide the same flexibility described for the row pattern.


#### C2.3 HTML Tag Removal

In [39]:
tag_regex = re.compile(r'<[^>]+>')

**Explanation:**

- `<` marks the beginning of an HTML tag.
- `[^>]+` matches one or more characters that are not `>`.
- `>` marks the end of the tag.
- Therefore, `<[^>]+>` matches an HTML tag so it can be removed from the extracted cell text.
- This is useful when the cell contains nested HTML formatting rather than plain text.

This removes HTML tags while keeping the text contained inside them.

#### C2.4 Whitespace Normalization

In [40]:
space_regex = re.compile(r'\s+')

**Explanation:**

`\s+` matches one or more whitespace characters, including spaces, tabs, and line breaks. Replacing them with a single space makes the extracted names and locations cleaner and more consistent.

#### C2.5 Selecting the Name and Location

After extracting the `<td>` cells, the code cleans every cell and keeps only non-empty cells. The first non-empty cell is assigned as the subsidiary name, while the last non-empty cell is assigned as the location.

This is important for this filing because the table contains empty cells used for spacing. Therefore, the code does not simply assume that the second physical `<td>` is always the location.

In [43]:
# --- Verification of Awkward Entries ---
print("Verification of awkward entries:")
print("="*60)

test_entries = [
    ("OSW Equipment & Repair, LLC", "Washington"),
    ("Truck Bodies & Equipment International, Inc.", "Delaware"),
    ("Victor Industrial Equipment (PTY) Limited", "South Africa"),
    ("Crysteel Manufacturing, Inc.", "Minnesota"),  # first
    ("Work Equipment Ltd.", "Canada")  # last
]

for name, loc in test_entries:
    found = (name, loc) in pairs  # ← pairs, bukan extracted_pairs
    status = "✓ FOUND" if found else "✗ MISSING"
    print(f"{status:10} | {name:45} | {loc}")

print("="*60)
print(f"Total extracted pairs: {len(pairs)}")

Verification of awkward entries:
✓ FOUND    | OSW Equipment & Repair, LLC                   | Washington
✓ FOUND    | Truck Bodies & Equipment International, Inc.  | Delaware
✓ FOUND    | Victor Industrial Equipment (PTY) Limited     | South Africa
✓ FOUND    | Crysteel Manufacturing, Inc.                  | Minnesota
✓ FOUND    | Work Equipment Ltd.                           | Canada
Total extracted pairs: 33


### C3. Reflection

The AI-generated regex worked correctly on the Federal Signal Corporation FY2024 Exhibit 21, so I did not find a failure that needed to be fixed. The verification process was important because I could confirm the result by comparing the extracted count with my manual ground truth and by checking the first, last, and awkward rows. Even if I could not understand the regex syntax, I could still detect a problem by comparing the AI output with the original filing and checking whether any subsidiaries were missing or incorrectly extracted. I would also test the same approach on a differently structured Exhibit 21 because a regex that works for one HTML structure may not work for every SEC filing.

## Bonus — Alternative Approach: BeautifulSoup

**Goal:** extract the same data using HTML parsing instead of regex, compare methods, and decide which is better.

In [45]:
# ===== BONUS: BeautifulSoup Alternative =====

from bs4 import BeautifulSoup

def extract_subsidiaries_beautifulsoup(html_content):
    """Extract subsidiaries using BeautifulSoup instead of regex."""
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find the table
    table = soup.find('table')
    
    bs_pairs = []
    
    if table:
        rows = table.find_all('tr')
        
        for row in rows[1:]:  # Skip header row
            cells = row.find_all('td')
            
            if len(cells) >= 2:
                # Extract text from first cell (subsidiary name)
                name_cell = cells[0].get_text(strip=True)
                
                # Extract text from last cell (location)
                location_cell = cells[-1].get_text(strip=True)
                
                # Filter out header-like rows
                if name_cell.lower() not in ['name of subsidiary', 'subsidiary name'] and \
                   location_cell.lower() not in ['jurisdiction of incorporation', 'state']:
                    
                    # Skip empty rows
                    if name_cell and location_cell:
                        bs_pairs.append((name_cell, location_cell))
    
    return bs_pairs

# Run BeautifulSoup extraction
bs_pairs = extract_subsidiaries_beautifulsoup(html)

print("=== BeautifulSoup Extraction Results ===\n")
print(f"BeautifulSoup extracted: {len(bs_pairs)} pairs")
print(f"Regex extracted: {len(pairs)} pairs\n")

# Compare the two methods
print("First 10 results (side by side):\n")
print(f"{'#':<3} {'Regex':<45} {'BeautifulSoup':<45}")
print("-" * 95)

for i in range(min(10, len(pairs), len(bs_pairs))):
    regex_result = pairs[i]
    bs_result = bs_pairs[i] if i < len(bs_pairs) else ("N/A", "N/A")
    
    regex_str = f"{regex_result[0][:40]}" if len(regex_result[0]) <= 40 else f"{regex_result[0][:37]}..."
    bs_str = f"{bs_result[0][:40]}" if len(bs_result[0]) <= 40 else f"{bs_result[0][:37]}..."
    
    print(f"{i+1:<3} {regex_str:<45} {bs_str:<45}")

# Full comparison
print("\n=== Complete Comparison ===\n")

if pairs == bs_pairs:
    print("✅ IDENTICAL RESULTS: Both methods extracted exactly the same data!")
    print(f"   Count: {len(pairs)} pairs")
    print(f"   Content match: 100%")
else:
    print("⚠️ DIFFERENCES FOUND between Regex and BeautifulSoup:")
    
    # Find differences
    only_in_regex = set(pairs) - set(bs_pairs)
    only_in_bs = set(bs_pairs) - set(pairs)
    
    if only_in_regex:
        print(f"\n  Only in Regex ({len(only_in_regex)}):")
        for pair in list(only_in_regex)[:3]:
            print(f"    - {pair}")
    
    if only_in_bs:
        print(f"\n  Only in BeautifulSoup ({len(only_in_bs)}):")
        for pair in list(only_in_bs)[:3]:
            print(f"    - {pair}")

print(f"\nTotal extracted by Regex: {len(pairs)}")
print(f"Total extracted by BeautifulSoup: {len(bs_pairs)}")

=== BeautifulSoup Extraction Results ===

BeautifulSoup extracted: 33 pairs
Regex extracted: 33 pairs

First 10 results (side by side):

#   Regex                                         BeautifulSoup                                
-----------------------------------------------------------------------------------------------
1   Crysteel Manufacturing, Inc.                  Crysteel Manufacturing, Inc.                 
2   Deist Industries, LLC                         Deist Industries, LLC                        
3   Elgin Sweeper Company                         Elgin Sweeper Company                        
4   Federal Signal of Texas Corp.                 Federal Signal of Texas Corp.                
5   Federal Signal UK Holdings Limited            Federal Signal UK Holdings Limited           
6   Federal Signal VAMA, S.A.                     Federal Signal VAMA, S.A.                    
7   FS Depot, LLC                                 FS Depot, LLC                                

### Bonus Reflection: Which Method to Ship?

**Regex Method:**
- ✅ Pros: No external dependencies, fast, works offline
- ❌ Cons: Harder to understand and maintain, brittle to HTML variations

**BeautifulSoup Method:**
- ✅ Pros: Robust to HTML variations, more readable, designed for parsing HTML
- ❌ Cons: External dependency required, slightly slower

**Which would you ship and why?**

I would ship the **BeautifulSoup method** for production because:

1. **Maintainability**: The code is self-documenting — `find('table')` and `find_all('tr')` clearly express intent
2. **Robustness**: HTML parsing libraries handle edge cases like malformed tags, nested elements, and whitespace better than regex patterns
3. **Scalability**: If SEC changes their Exhibit 21 HTML structure slightly, BeautifulSoup adapts better than a fragile regex pattern
4. **Industry standard**: Professional web scraping uses dedicated HTML parsers, not regex

However, for this assignment's constraint of using regex, the regex solution demonstrates that I can verify and validate AI-generated code which was the learning goal.